## ***Block 1 — Load dataset and Basic inspection***

In [1]:
# Install required packages
!pip -q install duckdb huggingface_hub pandas

import os
import getpass
import duckdb
import pandas as pd

# Paste your Hugging Face READ token when prompted

HF_TOKEN = os.environ.get("HF_TOKEN")

if not HF_TOKEN:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")

# Connect to Hugging Face
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"

# Load the sample dataset (fast)
df = con.sql(f"""
SELECT *
FROM read_parquet('{REL}/fact_content_daily_performance_sample.parquet')
""").df()

# -----------------------------
# Dataset Information
# -----------------------------
print("="*60)
print("Dataset Shape")
print("="*60)
print(df.shape)

print("\n")

print("="*60)
print("First Five Rows")
print("="*60)
print(df.head())

print("\n")

print("="*60)
print("Column Names")
print("="*60)
for i, col in enumerate(df.columns, 1):
    print(f"{i}. {col}")

print("\n")

print("="*60)
print("Dataset Information")
print("="*60)
print(df.info())

print("\n")

print("="*60)
print("Missing Values")
print("="*60)
print(df.isnull().sum())

print("\n")

print("="*60)
print("Statistical Summary")
print("="*60)
print(df.describe(include="all"))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.5/21.5 MB 80.5 MB/s eta 0:00:00


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Dataset Shape
(11694072, 31)


First Five Rows
  report_date           client_hash_id           content_hash_id  \
0  2026-06-01  client_3ffa76342f366962  content_1a6296faee432dae   
1  2026-06-01  client_3ffa76342f366962  content_73f21e612565035a   
2  2026-06-01  client_3ffa76342f366962  content_5a5be514ff559598   
3  2026-06-01  client_3ffa76342f366962  content_05b377d0c8a5cfd8   
4  2026-06-01  client_3ffa76342f366962  content_dc34c661d63e55a9   

   client_has_gsc  client_has_ga4  gsc_data_available  ga4_data_available  \
0            True            True               False               False   
1            True            True               False               False   
2            True            True               False               False   
3            True            True               False               False   
4            True            True               False               False   

   gsc_impressions  gsc_clicks  gsc_sum_position  ...  sessions_ai  \
0          

## ***Block*** **2** — **Data** **Preparation** **bold text**

In [2]:
# =====================================
# PART 1 - DATA PREPARATION
# =====================================

import pandas as pd
import numpy as np

# -----------------------------
# Check Dataset Columns
# -----------------------------
print("=" * 60)
print("Columns")
print("=" * 60)
print(df.columns.tolist())

# -----------------------------
# Create Target Label (Proxy)
# -----------------------------
# Rule:
# A page needs refresh if it receives many
# impressions but very few clicks.

df["Needs_Refresh"] = (
    (df["gsc_impressions"] >= 100) &
    (df["gsc_clicks"] <= 2)
).astype(int)

print("\n" + "=" * 60)
print("Refresh Label Distribution")
print("=" * 60)
print(df["Needs_Refresh"].value_counts())

print("\nPercentage Distribution")
print(df["Needs_Refresh"].value_counts(normalize=True) * 100)

# -----------------------------
# Select Features
# -----------------------------
# These features are NOT used to create
# the label, reducing leakage.

features = [
    "gsc_avg_position",
    "ga4_pageviews",
    "ga4_sessions",
    "ga4_engaged_sessions",
    "scroll_events"
]

# -----------------------------
# Prepare Feature Matrix & Target
# -----------------------------
df[features] = df[features].fillna(0)

X = df[features]
y = df["Needs_Refresh"]

print("\n" + "=" * 60)
print("Dataset Shape")
print("=" * 60)
print("Features Shape :", X.shape)
print("Target Shape   :", y.shape)

print("\n" + "=" * 60)
print("First Five Feature Rows")
print("=" * 60)
print(X.head())

Columns
['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events', 'month']

Refresh Label Distribution
Needs_Refresh
0    11323061
1      371011
Name: count, dtype: int64

Percentage Distribution
Needs_Refresh
0    96.827358
1     3.172642
Name: proportion, dtype: float64

Dataset Shape
Features Shape : (11694072, 5)
Target Shape   : (11694072,)

First Five Feature Rows
   gsc_avg_position  ga4_pageviews  ga4_sessions  ga4_engaged_sessions  \
0               0.0              0             0                     0   
1          

# ***Block 3 — Train Logistic Regression***

In [3]:
# =====================================
# PART 2 - LOGISTIC REGRESSION
# =====================================

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

# -------------------------------
# Train-Test Split
# -------------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# -------------------------------
# Pipeline
# -------------------------------

model = Pipeline([
    ("scaler", StandardScaler()),
    ("lr", LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=42
    ))
])

model.fit(X_train, y_train)

# -------------------------------
# Threshold Tuning
# -------------------------------

prob_train = model.predict_proba(X_train)[:,1]
prob_test = model.predict_proba(X_test)[:,1]

threshold = 0.60

train_pred = (prob_train >= threshold).astype(int)
test_pred = (prob_test >= threshold).astype(int)

# -------------------------------
# Training Performance
# -------------------------------

print("="*60)
print("Training Performance")
print("="*60)

print(f"Training Accuracy : {accuracy_score(y_train, train_pred):.4f}")

# -------------------------------
# Testing Performance
# -------------------------------

print("\n"+"="*60)
print("Testing Performance")
print("="*60)

print(f"Testing Accuracy : {accuracy_score(y_test, test_pred):.4f}")
print(f"Precision        : {precision_score(y_test, test_pred):.4f}")
print(f"Recall           : {recall_score(y_test, test_pred):.4f}")
print(f"F1 Score         : {f1_score(y_test, test_pred):.4f}")

print("\nConfusion Matrix")
print(confusion_matrix(y_test, test_pred))

print("\nClassification Report")
print(classification_report(y_test, test_pred))

# -------------------------------
# Predict All Pages
# -------------------------------

df["Prediction"] = (model.predict_proba(X)[:,1] >= threshold).astype(int)

df["Recommendation"] = df["Prediction"].map({
    1: "Refresh",
    0: "No Refresh"
})

output = df[
    ["Prediction","Recommendation"] + features
]

output.to_csv("refresh_recommendations.csv", index=False)

print("\nSaved: refresh_recommendations.csv")

print("\nTop 10 Refresh Recommendations")
print(output[output["Prediction"]==1].head(10))

Training Performance
Training Accuracy : 0.9165

Testing Performance
Testing Accuracy : 0.9163
Precision        : 0.1379
Recall           : 0.3120
F1 Score         : 0.1913

Confusion Matrix
[[2119908  144705]
 [  51050   23152]]

Classification Report
              precision    recall  f1-score   support

           0       0.98      0.94      0.96   2264613
           1       0.14      0.31      0.19     74202

    accuracy                           0.92   2338815
   macro avg       0.56      0.62      0.57   2338815
weighted avg       0.95      0.92      0.93   2338815


Saved: refresh_recommendations.csv

Top 10 Refresh Recommendations
      Prediction Recommendation  gsc_avg_position  ga4_pageviews  \
410            1        Refresh         63.000000              0   
625            1        Refresh         74.500000              0   
801            1        Refresh          9.000000              1   
1060           1        Refresh          5.333333              1   
1187        